# Lab 3: Compiling Models for Dataflow Execution

**Course**: *Dataflow Architectures for AI: From Principles to Practice*  
**Week**: 3, Compilation for Dataflow  
**Instructor**: Dr. Kaoutar El Maghraoui (IBM Research)  
**Runtime**: Google Colab, GPU runtime required (T4)  
**Expected time**: 90–120 minutes

---

## Learning objectives
By the end of this lab you will be able to:
1. Compile a model with torch.compile and measure the speedup over eager mode.
2. Ablate the eager / aot_eager / inductor backends to locate where the speedup comes from.
3. Detect, count, and fix graph breaks that prevent whole-graph compilation.
4. Read the generated Triton vs cuBLAS kernels and relate fusion to dataflow.

## Prerequisites
Labs 1-2 complete, plus the Week 3 lecture on the compilation pipeline, graph breaks, and fusion.

## How this lab is graded

This lab includes **checkpoint cells for learner self-checks**: assertion cells inside each Part that verify your outputs land in the expected range as you work. They help you catch mistakes; they do **not** assign a grade.

Official course progress and certificate eligibility are tracked through the **Cognitive Class module review questions and the final exam**, not through this notebook. The analysis prompts in Part 5 are practice for the Module 3 review questions.

### Working through this lab
1. Run each cell in order. Make sure every **Checkpoint** cell passes (no `AssertionError`).
2. Fill in every `TODO` and answer every `YOUR ANALYSIS HERE` prompt.
3. When you are done, complete the **Module 3 review questions** on Cognitive Class.


## Environment Preflight

This cell detects your runtime environment and sets the execution mode:
- **Full GPU**: all exercises run on CUDA (recommended: T4 or better in Colab)
- **Reduced CPU**: GPU-dependent cells use CPU fallback with reduced workloads
- **Static read-only**: no execution; pre-computed outputs are displayed

In [ ]:
# === Environment Preflight ===
import sys
import importlib

# Check Python version
assert sys.version_info >= (3, 9), f"Python >= 3.9 required, got {sys.version}"

# Check PyTorch
try:
    import torch
    _torch_version = tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])
    assert _torch_version >= (2, 2), f"PyTorch >= 2.2 required, got {torch.__version__}"
except ImportError:
    raise RuntimeError("PyTorch not installed. Run: pip install torch>=2.2")

# Detect execution mode
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    MODE = "full-gpu"
    _device_name = torch.cuda.get_device_name(0)
else:
    DEVICE = torch.device("cpu")
    MODE = "reduced-cpu"
    _device_name = "CPU only"

print(f"╔══════════════════════════════════════════════════╗")
print(f"║  Environment Preflight                          ║")
print(f"╠══════════════════════════════════════════════════╣")
print(f"║  Python:    {sys.version.split()[0]:<37s}║")
print(f"║  PyTorch:   {torch.__version__:<37s}║")
print(f"║  Device:    {_device_name:<37s}║")
print(f"║  Mode:      {MODE:<37s}║")
print(f"╚══════════════════════════════════════════════════╝")

if MODE == "reduced-cpu":
    print("\n⚠️  No GPU detected. GPU-dependent cells will use CPU fallback.")
    print("   For full experience: Runtime → Change runtime type → T4 GPU")


---
## Setup

In [ ]:
!pip install -q transformers==4.44.2

import os
import time
import torch
import torch.nn as nn
import torch._dynamo as dynamo

assert torch.cuda.is_available(), "GPU runtime required. Runtime → Change runtime type → GPU."
device = torch.device("cuda")
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA: {torch.version.cuda}")

torch.manual_seed(42)

In [ ]:
# PyTorch version compatibility check
import torch
print(f'PyTorch version: {torch.__version__}')
assert int(torch.__version__.split('.')[0]) >= 2, (
    f'This lab requires PyTorch 2.x or later. Got {torch.__version__}. '
    f'Run: pip install --upgrade torch'
)
print('✓ PyTorch version compatible')


---
## Model Definition

We use a minimal transformer block that exercises the full compilation pipeline:
attention → layer norm → FFN → residual. Small enough to compile quickly on Colab.

In [ ]:
class MiniTransformerBlock(nn.Module):
    """A clean transformer block suitable for torch.compile."""
    def __init__(self, d_model=512, nhead=8, dim_ff=2048):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_ff),
            nn.GELU(),
            nn.Linear(dim_ff, d_model),
        )

    def forward(self, x):
        # Pre-norm architecture
        normed = self.norm1(x)
        attn_out, _ = self.attn(normed, normed, normed)
        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x


model = MiniTransformerBlock(d_model=512, nhead=8, dim_ff=2048).cuda().eval()
x = torch.randn(4, 64, 512, device=device)  # batch=4, seq_len=64, d_model=512

# Verify eager forward works
with torch.no_grad():
    eager_out = model(x)
print(f"Model output shape: {eager_out.shape}")
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

---
## Part 1: Compile and Measure Speedup (2 pts)

Use `torch.compile` to compile the model, then measure the speedup over eager execution.

**Important**: The first call to a compiled model triggers compilation (slow). Subsequent calls
use the cached compiled code (fast). Always warm up before measuring.

### Measurement Best Practices

> **⚠️ Single-run timings can be misleading.** GPU performance varies due to thermal throttling,
> background processes, and memory state. Always use multiple measurements.

**Protocol for reliable benchmarks:**
1. **Warmup:** 3-5 iterations (discarded) to fill caches and trigger JIT compilation
2. **Repeat:** N ≥ 10 measurement iterations
3. **Report:** Median (robust to outliers) ± IQR or percentiles (P5, P50, P95)
4. **Synchronize:** Always call `torch.cuda.synchronize()` before timing GPU ops
5. **Isolate:** Use `torch.inference_mode()` or `torch.no_grad()` to avoid autograd overhead


In [ ]:
# TODO: Implement a robust benchmarking utility
# Requirements:
# 1. Accept a callable fn and run it n_warmup times (discard results)
# 2. Run fn n_measure times, recording wall-clock time for each
# 3. Return statistics: median, mean, std, P5, P95, IQR
# 4. Handle CUDA synchronization (torch.cuda.synchronize() before/after)
#
# Hint: Use time.perf_counter() for high-resolution timing
# Hint: np.percentile(times, [5, 50, 95]) gives you P5, median, P95

import time
import numpy as np

def bench_robust(fn, n_warmup=5, n_measure=20, sync_cuda=True):
    """Run fn() with warmup, return timing statistics in milliseconds."""
    # YOUR CODE HERE
    pass

# Test your implementation:
# x = torch.randn(1024, 1024, device=DEVICE)
# stats = bench_robust(lambda: torch.mm(x, x))
# assert 'median_ms' in stats
# assert stats['n'] == 20


In [ ]:
# TODO: Compile the model with torch.compile (default settings)
compiled_model = None  # TODO: torch.compile(model)

# Warm up compiled model (triggers compilation)
print("Compiling (first call triggers JIT)...")
with torch.no_grad():
    _ = compiled_model(x)  # This is slow. compilation happens here
    torch.cuda.synchronize()
print("Compilation complete.")

# TODO: Measure eager latency (average of 50 runs)
def measure_latency(fn, input_tensor, n_runs=50, warmup=10):
    """Measure average latency in ms."""
    with torch.no_grad():
        for _ in range(warmup):
            fn(input_tensor)
        torch.cuda.synchronize()
        
        start = time.perf_counter()
        for _ in range(n_runs):
            fn(input_tensor)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
    return (elapsed / n_runs) * 1000  # ms

eager_ms = measure_latency(model, x)
compiled_ms = measure_latency(compiled_model, x)
speedup = eager_ms / compiled_ms

print(f"\nEager latency:    {eager_ms:.3f} ms")
print(f"Compiled latency: {compiled_ms:.3f} ms")
print(f"Speedup:          {speedup:.2f}x")

In [ ]:
# --- Checkpoint 1 ---
assert compiled_model is not None, "Compile the model with torch.compile()"
assert compiled_ms > 0, "Compiled latency must be measured"
assert speedup > 0.8, f"Expected some speedup or at least parity, got {speedup:.2f}x"

# Verify correctness: compiled output should match eager output
with torch.no_grad():
    compiled_out = compiled_model(x)
max_diff = (eager_out - compiled_out).abs().max().item()
assert max_diff < 1e-3, f"Compiled output diverges from eager by {max_diff:.6f}"

print(f"[OK] Checkpoint 1 passed!")
print(f"  Speedup: {speedup:.2f}x")
print(f"  Max numerical difference: {max_diff:.2e} (acceptable for FP32)")

### Compilation Latency vs. Steady-State Performance

`torch.compile` has a one-time cost: the first call triggers tracing, optimization, and code generation.
After that, subsequent calls run the optimized code at steady-state speed.

Three metrics matter for production decisions:
1. **First-call latency**: includes compilation (can be 10–60s for large models)
2. **Steady-state latency**: optimized execution (the payoff)
3. **Break-even count**: how many forward passes before compilation cost is recouped

In [ ]:
import time

# TODO: Measure compilation vs steady-state performance
# 1. Reset dynamo and create a compiled model
# 2. Time the FIRST call (this includes compilation)
# 3. Run N_WARMUP=3 warmup iterations (discard)
# 4. Time N_MEASURE=20 steady-state iterations
# 5. Time the eager baseline for the same N_MEASURE iterations
# 6. Calculate break-even = compilation_overhead / savings_per_call

torch._dynamo.reset()
# model_compile = torch.compile(model)

N_WARMUP, N_MEASURE = 3, 20

# YOUR CODE HERE
# first_call_ms = ...
# steady_state_ms = ...
# eager_ms = ...
# break_even = ...

# print(f"First-call (includes compilation): {first_call_ms:>10.1f} ms")
# print(f"Steady-state (compiled):           {steady_state_ms:>10.2f} ms")  
# print(f"Eager baseline:                    {eager_ms:>10.2f} ms")
# print(f"Break-even iterations:             {break_even:>10d}")

---
## Part 2: Backend Ablation (2 pts)

PyTorch's compilation stack has multiple stages. By changing the `backend` argument,
you can stop at different levels and isolate which stage provides the speedup:

| Backend | What it tests |
|---------|---------------|
| `"eager"` | Only TorchDynamo graph capture (no optimization) |
| `"aot_eager"` | + AOTAutograd (backward graph generation) |
| `"inductor"` | Full stack: + TorchInductor (Triton codegen, operator fusion) |

If a bug appears with `"inductor"` but not `"aot_eager"`, the bug is in TorchInductor.

In [ ]:
# TODO: Compile with each backend and measure latency
# Reset dynamo between compilations to avoid cached graphs

backends = ["eager", "aot_eager", "inductor"]
results = {}

for backend in backends:
    dynamo.reset()  # Clear compilation cache
    
    # TODO: Compile with this backend
    compiled = None  # TODO: torch.compile(model, backend=backend)
    
    # Warm up
    with torch.no_grad():
        out = compiled(x)
        torch.cuda.synchronize()
    
    # Measure
    latency = measure_latency(compiled, x)
    
    # Check correctness vs eager
    with torch.no_grad():
        test_out = compiled(x)
    diff = (eager_out - test_out).abs().max().item()
    
    results[backend] = {"latency_ms": latency, "max_diff": diff}
    print(f"Backend '{backend}': {latency:.3f} ms, max_diff={diff:.2e}")

print(f"\nInductor speedup over eager backend: {results['eager']['latency_ms']/results['inductor']['latency_ms']:.2f}x")
print(f"AOT-eager speedup over eager backend: {results['eager']['latency_ms']/results['aot_eager']['latency_ms']:.2f}x")

In [ ]:
# --- Checkpoint 2 ---
assert len(results) == 3, "Must test all three backends"
assert all(r["max_diff"] < 1e-2 for r in results.values()), "All backends must produce correct output"
assert results["inductor"]["latency_ms"] <= results["eager"]["latency_ms"] * 1.5, \
    "Inductor should not be dramatically slower than eager"

print("[OK] Checkpoint 2 passed!")
print(f"  Key insight: the speedup comes from {'inductor (fusion/codegen)' if results['inductor']['latency_ms'] < results['aot_eager']['latency_ms'] else 'aot_eager (graph capture)'}")

---
## Part 3: Graph Break Detection (2 pts)

A **graph break** occurs when TorchDynamo encounters Python code it cannot capture
into the computation graph. The model still runs, but the broken subgraphs fall back
to eager execution, silently losing the compilation benefit.

Below is a model with **deliberate graph breaks**. Your job: detect and count them.

In [ ]:
class BrokenTransformerBlock(nn.Module):
    """Same as MiniTransformerBlock but with graph-breaking operations."""
    def __init__(self, d_model=512, nhead=8, dim_ff=2048):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_ff),
            nn.GELU(),
            nn.Linear(dim_ff, d_model),
        )
        self.call_count = 0

    def forward(self, x):
        # --- Graph Break 1: .item() forces CPU sync ---
        max_val = x.max().item()
        if max_val > 1000:  # never true, but dynamo can't prove it
            x = x * 0.5

        normed = self.norm1(x)
        attn_out, _ = self.attn(normed, normed, normed)
        x = x + attn_out

        # --- Graph Break 2: print() is not traceable ---
        self.call_count += 1
        print(f"Forward call #{self.call_count}", end="\r")

        x = x + self.ffn(self.norm2(x))

        # --- Graph Break 3: numpy conversion ---
        norm_val = x.norm().cpu().numpy()
        _ = norm_val  # unused but still breaks the graph

        return x


broken_model = BrokenTransformerBlock().cuda().eval()

In [ ]:
# TODO: Use torch._dynamo.explain() to detect graph breaks
# This function analyzes the model and reports:
#   - Number of graph breaks
#   - Reasons for each break
#   - Number of resulting subgraphs

dynamo.reset()

# TODO: Call dynamo.explain on the broken model
explanation = None  # TODO: dynamo.explain(broken_model)(x)

# Print the explanation
print(explanation)

# TODO: Extract the number of graph breaks
n_graph_breaks = None  # TODO: extract from explanation (explanation.graph_break_count)
n_graphs = None        # TODO: extract from explanation (explanation.graph_count)

print(f"\n--- Summary ---")
print(f"Graph breaks: {n_graph_breaks}")
print(f"Resulting subgraphs: {n_graphs}")
print(f"Each break = a CPU sync + Python fallback + re-entry into compiled code")

In [ ]:
# Measure the performance impact of graph breaks
dynamo.reset()
compiled_broken = torch.compile(broken_model)

# Warm up (suppress print output)
import io, contextlib
with torch.no_grad(), contextlib.redirect_stdout(io.StringIO()):
    for _ in range(5):
        compiled_broken(x)
    torch.cuda.synchronize()

# Measure
with contextlib.redirect_stdout(io.StringIO()):
    broken_ms = measure_latency(compiled_broken, x)

print(f"Clean model (compiled):  {compiled_ms:.3f} ms")
print(f"Broken model (compiled): {broken_ms:.3f} ms")
print(f"Slowdown from breaks:    {broken_ms/compiled_ms:.2f}x")

In [ ]:
# --- Checkpoint 3 ---
assert n_graph_breaks is not None, "Extract the graph break count from explanation"
assert n_graph_breaks >= 2, f"Expected at least 2 graph breaks, found {n_graph_breaks}"
assert n_graphs is not None, "Extract the graph count"
assert n_graphs > n_graph_breaks, "Number of graphs should exceed number of breaks (graphs = breaks + 1)"

print(f"[OK] Checkpoint 3 passed!")
print(f"  Detected {n_graph_breaks} graph breaks → {n_graphs} subgraphs")
print(f"  Performance penalty: {broken_ms/compiled_ms:.1f}x slower than clean compilation")

---
## Part 4: Fix Graph Breaks (2 pts)

Now fix `BrokenTransformerBlock` to eliminate all graph breaks.

**Rules for fixing**:
- Remove or replace `.item()` calls. Use tensor operations instead
- Remove `print()` statements (or use `torch._dynamo.config.suppress_errors`)
- Remove `.numpy()` conversions. Stay in tensor land
- Do NOT change the mathematical behavior of the model

In [ ]:
class FixedTransformerBlock(nn.Module):
    """TODO: Fix the graph breaks from BrokenTransformerBlock.
    
    Requirements:
    - Same mathematical behavior (attention + FFN + residuals)
    - Zero graph breaks when compiled
    - Numerically close to eager execution (max_diff < 1e-3)
    """
    def __init__(self, d_model=512, nhead=8, dim_ff=2048):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_ff),
            nn.GELU(),
            nn.Linear(dim_ff, d_model),
        )

    def forward(self, x):
        # TODO: Implement the same logic WITHOUT graph breaks
        # Hints:
        #   - Replace .item() + if with torch.where() or just remove the dead branch
        #   - Remove print() entirely
        #   - Remove .numpy() conversion
        raise NotImplementedError("Fix the graph breaks")

In [ ]:
# Validate the fix
dynamo.reset()

fixed_model = FixedTransformerBlock().cuda().eval()
# Copy weights from original clean model for fair comparison
fixed_model.load_state_dict(model.state_dict())

# Check: zero graph breaks
fixed_explanation = dynamo.explain(fixed_model)(x)
fixed_breaks = fixed_explanation.graph_break_count
print(f"Graph breaks in fixed model: {fixed_breaks}")

# Compile and measure
dynamo.reset()
compiled_fixed = torch.compile(fixed_model)
with torch.no_grad():
    _ = compiled_fixed(x)  # trigger compilation
    torch.cuda.synchronize()

fixed_ms = measure_latency(compiled_fixed, x)

# Verify correctness
with torch.no_grad():
    fixed_out = compiled_fixed(x)
fix_diff = (eager_out - fixed_out).abs().max().item()

print(f"\nFixed model latency: {fixed_ms:.3f} ms")
print(f"Speedup vs broken:   {broken_ms/fixed_ms:.2f}x")
print(f"Max diff vs eager:   {fix_diff:.2e}")

In [ ]:
# --- Checkpoint 4 ---
assert fixed_breaks == 0, f"Fixed model still has {fixed_breaks} graph breaks. Keep fixing!"
assert fix_diff < 1e-3, f"Fixed model output diverges too much: {fix_diff:.6f}"
assert fixed_ms < broken_ms * 1.1, "Fixed model should be at least as fast as broken version"

print(f"[OK] Checkpoint 4 passed!")
print(f"  Zero graph breaks achieved!")
print(f"  Latency: {fixed_ms:.3f} ms (vs {broken_ms:.3f} ms broken, {compiled_ms:.3f} ms original)")

---
## Part 5: Inspect Generated Code (2 pts)

Use `TORCH_COMPILE_DEBUG=1` to dump the compilation artifacts, then examine
the generated Triton kernel code to identify operator fusion.

In [ ]:
# Enable TORCH_COMPILE_DEBUG to dump artifacts
os.environ["TORCH_COMPILE_DEBUG"] = "1"

dynamo.reset()
debug_model = torch.compile(MiniTransformerBlock().cuda().eval())

with torch.no_grad():
    _ = debug_model(x)
    torch.cuda.synchronize()

# Disable for subsequent cells
os.environ.pop("TORCH_COMPILE_DEBUG", None)

print("Debug artifacts written. Check torch_compile_debug/ directory.")

In [ ]:
# List the generated artifacts
import glob

debug_dirs = glob.glob("torch_compile_debug/run_*")
if debug_dirs:
    latest = sorted(debug_dirs)[-1]
    print(f"Latest debug directory: {latest}")
    
    # Find generated output code
    output_files = glob.glob(f"{latest}/**/output_code.py", recursive=True)
    fx_files = glob.glob(f"{latest}/**/fx_graph_readable.py", recursive=True)
    
    print(f"\nFound {len(output_files)} output_code.py file(s)")
    print(f"Found {len(fx_files)} fx_graph_readable.py file(s)")
    
    # Show the FX graph (high-level IR)
    if fx_files:
        print(f"\n{'='*60}")
        print(f"FX GRAPH (high-level IR):")
        print(f"{'='*60}")
        with open(fx_files[0]) as f:
            content = f.read()
        # Show first 80 lines
        lines = content.split('\n')[:80]
        print('\n'.join(lines))
        if len(content.split('\n')) > 80:
            print(f"\n... ({len(content.split(chr(10)))} total lines)")
else:
    print("No debug directory found. TORCH_COMPILE_DEBUG may not have worked.")
    print("This can happen if the model was already cached. Try: dynamo.reset() first.")

In [ ]:
# Show generated Triton/C++ code (the actual kernel)
if debug_dirs and output_files:
    print(f"{'='*60}")
    print(f"GENERATED OUTPUT CODE (fused kernels):")
    print(f"{'='*60}")
    with open(output_files[0]) as f:
        output_content = f.read()
    # Show first 100 lines of generated code
    lines = output_content.split('\n')[:100]
    print('\n'.join(lines))
    
    # Count triton kernels (look for @triton.jit or def triton_)
    triton_kernels = output_content.count('def triton_')
    extern_kernels = output_content.count('extern_kernels.')
    print(f"\n--- Kernel Statistics ---")
    print(f"Triton (fused) kernels: {triton_kernels}")
    print(f"External kernel calls:  {extern_kernels}")
    print(f"Fusion ratio: {triton_kernels}/{triton_kernels + extern_kernels} kernels are fused")

In [ ]:
# TODO: Written analysis of compilation artifacts
compilation_analysis = """
TODO: Replace this with your analysis (3-5 sentences). Address:

1. How many separate kernels did Inductor generate? How does this compare to the
   number of PyTorch operators in the original model (count: 2 norms + 1 attention +
   2 linears + 1 GELU + 2 adds = ~8 ops)?

2. Which operators were FUSED into Triton kernels vs. which used external (cuBLAS/cuDNN)
   kernels? Why do you think the compiler made that choice?

3. How does this operator fusion relate to the DATAFLOW concept from lecture?
   (Hint: fusion eliminates intermediate memory writes; same as dataflow pipelining)
"""
print(compilation_analysis)

In [ ]:
# --- Checkpoint 5 ---
assert "TODO" not in compilation_analysis, "Write your analysis (replace the TODO block)"
assert len(compilation_analysis.strip()) > 100, "Analysis should be at least a few sentences."

print("[OK] Checkpoint 5 passed!")
print("  Compilation artifact analysis complete.")
print(f"\n{'='*60}")
print("LAB 3 COMPLETE")
print(f"{'='*60}")
print(f"Summary:")
print(f"  Section 1 - Speedup: {speedup:.2f}x")
print(f"  Section 2 - Best backend: inductor ({results['inductor']['latency_ms']:.3f} ms)")
print(f"  Section 3 - Graph breaks detected: {n_graph_breaks}")
print(f"  Section 4 - Breaks fixed: {fixed_breaks} remaining")
print(f"  Section 5 - Artifacts inspected")

---

## Part 6 - Update the Accelerator Landscape Map

Add this week's row to the shared map you have been building since Lab 1.


In [ ]:
# Part 6 -- Landscape Map row for Week 3. Fill the measured values from your run.
landscape_row_w3 = {
    "accelerator":               (f"GPU ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "CPU"),
    "graph_capture":             "TorchDynamo (torch.compile)",
    "lowering_pipeline":         "Dynamo -> AOTAutograd -> TorchInductor",
    "codegen":                   "Triton (fused elementwise/norm) + cuBLAS (GEMMs)",
    "eager_vs_compiled_speedup": None,   # TODO: round(speedup, 2)
    "graph_breaks_before_fix":   None,   # TODO: int(n_graph_breaks)
    "graph_breaks_after_fix":    None,   # TODO: int(fixed_breaks)
}
for _k, _v in landscape_row_w3.items():
    print(f"  {_k:27s}: {_v}")


---
## Optional: Spyre Compilation Deep Dive

*Requires Spyre access via IBM Research partnership.*

1. Compile the same `MiniTransformerBlock` for Spyre using `torch.compile(model, backend="sendnn")`
2. Set `TORCH_COMPILE_DEBUG=1` and examine the `sdsc_*.json` files (Spyre's backend IR)
3. Compare: How does Spyre's compilation artifact structure differ from Inductor's Triton output?
4. Identify which operators Spyre fuses vs. which it keeps as separate kernels

---

*Course: Dataflow Architectures for AI, Week 3 Lab, v1.0*

> **Spyre vocabulary (consistent across all labs):** `import torch_sendnn` registers the `aiu` device; compile with `torch.compile(model, backend="sendnn")`; dump the backend IR with `TORCH_COMPILE_DEBUG=1` (look for `sdsc_*.json`); raise verbosity with `SENDNN_LOG_LEVEL=DEBUG`. Requires Spyre access via an IBM Research partnership.
